# UFACTORY xArm 6

The [UFACTORY xArm 6](https://www.ufactory.cc/xarm-collaborative-robot/) is a 6-axis articulated robotic arm. PyLabRobot supports it with the xArm bio-gripper. It supports:

- [Arms](../../capabilities/arms) (Cartesian and joint movement, pick/place, freedrive teaching)

The device communicates over Ethernet using the [xarm-python-sdk](https://pypi.org/project/xarm-python-sdk/). Install the optional dependency group with `pip install PyLabRobot[xarm]`.

| Model | PLR Name | Notes |
|---|---|---|
| xArm 6 | `XArm6` | 6-DOF articulated arm + bio-gripper |

## Setup

Build a driver (IP, optional TCP offset/load), wrap it in `XArm6`, and call `setup()`.

In [ ]:
from pylabrobot.ufactory.xarm6 import XArm6, XArm6Driver

driver = XArm6Driver(
  ip="192.168.1.220",
  tcp_offset=(0, 0, 0, 0, 0, 0),  # adjust for your gripper mount (x, y, z, roll, pitch, yaw)
)
xarm = XArm6(driver=driver)
await xarm.setup()

`setup()` connects to the controller, clears any pending errors, enables motion, and initializes the bio-gripper. To skip the gripper init, pass `backend_params=XArm6Driver.SetupParams(skip_gripper_init=True)` to `setup()`.

## Arm capabilities

The xArm 6 exposes an {class}`~pylabrobot.capabilities.arms.articulated_arm.ArticulatedArm` on `xarm.arm`, backed by {class}`~pylabrobot.ufactory.xarm6.backend.XArm6ArmBackend`. For the full arm API, see [Arms](../../capabilities/arms).

### Gripper control

Gripper widths are in millimeters. The bio-gripper range is 71 mm (fully closed) to 150 mm (fully open).

In [ ]:
await xarm.arm.open_gripper(gripper_width=150)

In [ ]:
await xarm.arm.close_gripper(gripper_width=71)

In [ ]:
await xarm.arm.is_gripper_closed()

### Cartesian movement

Move the arm to a Cartesian location with a full 3D rotation. Use {class}`~pylabrobot.ufactory.xarm6.backend.XArm6ArmBackend.CartesianMoveParams` to override the default speed and acceleration.

In [ ]:
from pylabrobot.resources import Coordinate
from pylabrobot.resources.rotation import Rotation
from pylabrobot.ufactory.xarm6 import XArm6ArmBackend

await xarm.arm.move_to_location(
  location=Coordinate(x=300, y=0, z=200),
  rotation=Rotation(x=180, y=0, z=0),  # roll, pitch, yaw (degrees)
  backend_params=XArm6ArmBackend.CartesianMoveParams(speed=150, mvacc=2500),
)

### Joint movement

Move in joint space using the {class}`~pylabrobot.ufactory.xarm6.joints.XArm6Axis` enum and {class}`~pylabrobot.ufactory.xarm6.backend.XArm6ArmBackend.JointMoveParams`:

```{warning}
Moving to arbitrary joint positions can cause the arm to collide with its base, the gripper, or nearby equipment. Verify coordinates carefully in a safe workspace first.
```

In [ ]:
from pylabrobot.ufactory.xarm6 import XArm6Axis

position = {
  XArm6Axis.BASE_ROTATION: 0.0,
  XArm6Axis.SHOULDER: -30.0,
  XArm6Axis.ELBOW: -60.0,
  XArm6Axis.WRIST_ROLL: 0.0,
  XArm6Axis.WRIST_PITCH: 90.0,
  XArm6Axis.WRIST_YAW: 0.0,
}
await xarm.arm.backend.move_to_joint_position(
  position,
  backend_params=XArm6ArmBackend.JointMoveParams(speed=40),
)

### Querying position

Get the current joint angles or Cartesian end-effector pose:

In [ ]:
await xarm.arm.backend.request_joint_position()

In [ ]:
await xarm.arm.backend.request_gripper_location()

### Pick and place

`pick_up_at_location` moves the arm to the target pose and closes the gripper to `resource_width`. `drop_at_location` moves to the target and fully opens the gripper. Approach and retract motions are the caller's responsibility — wrap these calls with your own pre- and post-moves as needed.

In [ ]:
pick = Coordinate(x=300, y=100, z=50)
place = Coordinate(x=300, y=-100, z=50)
above = Coordinate(x=0, y=0, z=80)
down = Rotation(x=180, y=0, z=0)

Approach the pick position from 80 mm above, descend, close the gripper, and retract:

In [ ]:
await xarm.arm.move_to_location(pick + above, down)
await xarm.arm.pick_up_at_location(location=pick, rotation=down, resource_width=80)
await xarm.arm.move_to_location(pick + above, down)

Travel to the place position, descend, release, and retract:

In [ ]:
await xarm.arm.move_to_location(place + above, down)
await xarm.arm.drop_at_location(location=place, rotation=down)
await xarm.arm.move_to_location(place + above, down)

### Freedrive (teaching mode)

Enter freedrive mode to manually position the arm by hand, then read coordinates for use in your protocol. The xArm SDK frees all axes simultaneously; the `free_axes` argument is accepted for API compatibility but ignored.

In [ ]:
await xarm.arm.backend.start_freedrive_mode(free_axes=[0])

In [ ]:
# Manually guide the arm to the desired pose, then read it:
taught = await xarm.arm.backend.request_gripper_location()
print(taught)

In [ ]:
await xarm.arm.backend.stop_freedrive_mode()

### Park, halt, and error recovery

Park the arm (SDK home by default, or a user-supplied `park_location`), emergency-stop all motion, or clear an error state:

In [ ]:
await xarm.arm.backend.park()

In [ ]:
await xarm.arm.backend.halt()

In [ ]:
await xarm.driver.clear_errors()

## Teardown

In [ ]:
await xarm.stop()